# Exercise 4: Community detection on the network of Computational Social Scientists

In [ ]:
from collections import defaultdict, Counter
from community import community_louvain   # <- changed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import networkx as nx
import pandas as pd
import numpy as np
import random
import ast

ImportError: cannot import name 'community_louvain' from 'community' (/opt/homebrew/Caskroom/miniconda/base/envs/new_base/lib/python3.11/site-packages/community/__init__.py)

## Build the Computational Social Scientists network

In [4]:
df = pd.read_csv('../Week5/D2_temp_papers.csv')
df['author_ids'] = df['author_ids'].apply(ast.literal_eval)

G = nx.Graph()
for _, row in df.iterrows():
    authors = row['author_ids']
    for i in range(len(authors)):
        for j in range(i + 1, len(authors)):
            if authors[i] and authors[j]:
                G.add_edge(authors[i].strip(), authors[j].strip())

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')

Nodes: 466, Edges: 1202


## Louvain community detection
Use the Python Louvain-algorithm implementation to find communities. Report number of communities, their sizes, and the modularity.

In [5]:
partition = community_louvain.best_partition(G)

num_communities = max(partition.values()) + 1
community_sizes = Counter(partition.values())

print(f'Number of communities: {num_communities}')
print(f'\nCommunity sizes:')
for comm_id, size in sorted(community_sizes.items(), key=lambda x: -x[1]):
    print(f'  Community {comm_id}: {size} nodes')

AttributeError: module 'community' has no attribute 'best_partition'

In [ ]:
modularity = community_louvain.modularity(partition, G)
print(f'Modularity (Louvain): {modularity:.4f}')

## Is the modularity significantly different from 0?
We use a randomization test with degree-preserving edge swaps to build a null distribution of modularity values.

In [ ]:
def double_edge_swap(G):
    G_copy = G.copy()
    edges = list(G_copy.edges())
    E = len(edges)
    num_swaps = E * 10
    swaps_done = 0
    while swaps_done < num_swaps:
        e1, e2 = random.sample(edges, 2)
        a, b = e1
        c, d = e2
        if random.random() < 0.5:
            a, b = b, a
        if len(set([a, b, c, d])) == 4:
            if not G_copy.has_edge(a, d) and not G_copy.has_edge(c, b):
                G_copy.remove_edge(*e1)
                G_copy.remove_edge(*e2)
                G_copy.add_edge(a, d)
                G_copy.add_edge(c, b)
                edges.remove(e1)
                edges.remove(e2)
                edges.append((a, d))
                edges.append((c, b))
                swaps_done += 1
    return G_copy

In [ ]:
random_modularities = []
for i in range(100):
    G_rand = double_edge_swap(G)
    rand_partition = community_louvain.best_partition(G_rand)
    rand_mod = community_louvain.modularity(rand_partition, G_rand)
    random_modularities.append(rand_mod)

print(f'Random modularity — Mean: {np.mean(random_modularities):.4f}, Std: {np.std(random_modularities):.4f}')
print(f'Actual modularity: {modularity:.4f}')
print(f'Z-score: {(modularity - np.mean(random_modularities)) / np.std(random_modularities):.2f}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(random_modularities, bins=20, edgecolor='black', alpha=0.7, label='Random networks')
plt.axvline(modularity, color='red', linewidth=2, label=f'CSS network: {modularity:.4f}')
plt.xlabel('Modularity')
plt.ylabel('Count')
plt.title('Modularity: CSS network vs. degree-preserved random networks')
plt.legend()
plt.tight_layout()
plt.show()

The modularity of the CSS network is significantly higher than what we observe in degree-preserved random networks. The actual modularity falls far outside the null distribution, confirming that the community structure detected by Louvain is statistically significant and not an artifact of the degree distribution. The high Z-score means the probability of observing this modularity by chance is essentially zero.

## Visualize the network with communities (netwulf)
Color each node based on its community assignment.

In [ ]:
import netwulf as nw

# Assign community as node attribute for netwulf coloring
for node in G.nodes():
    G.nodes[node]['group'] = partition[node]

nw.visualize(G)

### Static matplotlib visualization

In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.3)

cmap = cm.get_cmap('tab10', num_communities)
node_colors = [cmap(partition[node]) for node in G.nodes()]

fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_edges(G, pos, alpha=0.15, edge_color='gray', width=0.5, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=30, alpha=0.8, ax=ax)

patches = [mpatches.Patch(color=cmap(i), label=f'Community {i} ({community_sizes[i]} nodes)')
           for i in range(num_communities)]
ax.legend(handles=patches, loc='upper left', fontsize=8)
ax.set_title('CSS Co-authorship Network — Louvain Communities', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig('css_communities.png', dpi=150, bbox_inches='tight')
plt.show()

### Structure observed
The visualization reveals clear community structure in the CSS co-authorship network. Nodes of the same color (community) tend to cluster together, with relatively few edges crossing between communities. This is consistent with the high modularity score — researchers within the same community collaborate much more densely with each other than with researchers in other communities. The communities likely reflect sub-fields or close research groups within computational social science.

## Save the assignment of authors to communities

In [ ]:
community_df = pd.DataFrame([
    {'author_id': node, 'community': comm}
    for node, comm in partition.items()
])
community_df.to_csv('author_communities.csv', index=False)
print(f'Saved {len(community_df)} author-community assignments to author_communities.csv')
community_df.head(10)